In [ ]:
"""
Continuous AUC(t) Curves for BrainVital8
==========================================
Compute and plot time-dependent cumulative AUC curves with bootstrap CI
for BrainVital8 vs comparator scores across cohorts.

Two analysis modes:
  - with_age:    BrainVital8_age vs age-inclusive comparators
  - without_age: BrainVital8 vs age-free comparators
  - Age shown as grey dashed benchmark in both

Outputs per mode:
  tables/AUC_curves_long.csv
  figures/fig_AUC_curves.pdf
"""

import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
from joblib import Parallel, delayed

from sksurv.util import Surv
from sksurv.metrics import cumulative_dynamic_auc


# ========================= Configuration =========================

PROJECT_DIR = Path("./data/work")
DATA_PATH   = PROJECT_DIR / "BrainVital8_all_cohorts.csv"
OUT_BASE    = "./results/modifiable_comparison_final"

time_col   = "time"
event_col  = "status"
cohort_col = "cohort"

AGE_LABEL, AGE_COL = "Age", "age"

ANALYSIS_CONFIGS = {
    "with_age": {
        "main": ("BrainVital8", "BrainVital8_age"),
        "comparators": [
            ("LIBRA2",   "libra2_0"),
            ("CAIDE",    "caide_0"),
            ("ANU-ADRI", "ANU_0"),
            ("CogDrisk", "CogD_0"),
            ("LE8",      "le8_0"),
        ],
        "description": "All scores adjusted for / include age",
    },
    "without_age": {
        "main": ("BrainVital8", "BrainVital8"),
        "comparators": [
            ("LIBRA2",   "libra2_0"),
            ("CAIDE",    "caide_0_noage"),
            ("ANU-ADRI", "ANU_0_noage"),
            ("CogDrisk", "CogD_0_noage"),
            ("LE8",      "le8_0"),
        ],
        "description": "All scores exclude age",
    },
}

# Scores where higher = healthier -> negate for risk direction
NEGATE_SCORES = {"BrainVital8", "BrainVital8_age", "le8_0"}

# Per-cohort curve parameters
COHORT_GRID = {
    "UKB":    dict(q_end=0.95, curve_start_y=3.0, curve_n=40),
    "ELSA":   dict(q_end=0.70, curve_start_y=5.0, curve_n=20),
    "SHARE":  dict(q_end=0.95, curve_start_y=3.0, curve_n=40),
    "HRS":    dict(q_end=0.95, curve_start_y=3.0, curve_n=40),
    "KLOSA":  dict(q_end=0.95, curve_start_y=3.0, curve_n=40),
    "CHARLS": dict(q_end=0.95, curve_start_y=3.0, curve_n=40),
}
DEFAULT_GRID = dict(q_end=0.95, curve_start_y=3.0, curve_n=40)

N_BOOT      = 1000
RANDOM_SEED = 42
N_JOBS      = -1

COLORS = {
    "BrainVital8": "#E63946",
    "Age":         "#6C757D",
    "LIBRA2":      "#457B9D",
    "CAIDE":       "#2A9D8F",
    "ANU-ADRI":    "#8FB8A0",
    "CogDrisk":    "#F4A261",
    "LE8":         "#1D3557",
}


# ========================= Utilities =========================

def coerce_numeric(df, cols):
    """Convert columns to numeric, coercing errors to NaN."""
    for c in cols:
        if c in df.columns:
            if df[c].dtype == "object":
                df[c] = df[c].astype(str).str.strip()
            df[c] = pd.to_numeric(df[c], errors="coerce")
    return df


def risk_array(df, col):
    """Get risk array, negating scores where higher = healthier."""
    raw = df[col].to_numpy(dtype=float)
    return -raw if col in NEGATE_SCORES else raw


def get_aligned(df_in, score_cols):
    """Drop rows with missing scores/time/event, keep time > 0."""
    valid = [c for c in score_cols if c in df_in.columns]
    sub = df_in.dropna(subset=valid + [time_col, event_col]).copy()
    sub = sub[sub[time_col] > 0]
    return sub, valid


# ========================= AUC Curve Computation =========================

def compute_auc_curve_with_ci(aligned, col, times_days, n_boot, seed, n_jobs):
    """Compute cumulative dynamic AUC at each time point with bootstrap CI."""
    time_arr  = aligned[time_col].to_numpy(dtype=float)
    event_arr = aligned[event_col].to_numpy(dtype=int).astype(bool)
    risk      = risk_array(aligned, col)
    n         = len(aligned)

    # Point estimate
    try:
        y = Surv.from_arrays(event=event_arr, time=time_arr)
        auc_obs, _ = cumulative_dynamic_auc(y, y, risk, times_days)
    except Exception as e:
        print(f"        [{col}] point estimate failed: {e}")
        return None

    # Bootstrap CI
    rng = np.random.default_rng(seed)
    all_idx = rng.integers(0, n, size=(n_boot, n))

    def one_boot(idx):
        try:
            y_b = Surv.from_arrays(event=event_arr[idx], time=time_arr[idx])
            auc_b, _ = cumulative_dynamic_auc(y_b, y_b, risk[idx], times_days)
            return auc_b
        except Exception:
            return None

    boot_results = Parallel(n_jobs=n_jobs, backend="loky")(
        delayed(one_boot)(all_idx[i]) for i in range(n_boot)
    )
    boot_aucs = np.array([r for r in boot_results if r is not None], dtype=float)

    if len(boot_aucs) < max(30, int(0.2 * n_boot)):
        return dict(auc=auc_obs, ci_low=None, ci_high=None)

    ci_low  = np.percentile(boot_aucs, 2.5, axis=0)
    ci_high = np.percentile(boot_aucs, 97.5, axis=0)
    return dict(auc=auc_obs, ci_low=ci_low, ci_high=ci_high)


# ========================= Per-Cohort Processing =========================

def process_cohort_curves(cohort_name, df_cohort, main_label, main_col, comparators):
    """Compute AUC curves for all scores in one cohort."""
    grid = COHORT_GRID.get(cohort_name, DEFAULT_GRID)
    all_scores = [(main_label, main_col)] + comparators
    score_cols = [c for _, c in all_scores]

    aligned, valid = get_aligned(df_cohort, score_cols)
    n_total  = len(aligned)
    n_events = int(aligned[event_col].sum()) if n_total else 0
    print(f"  [{cohort_name}] aligned n={n_total}, events={n_events}")

    if n_total < 200 or n_events < 30:
        print(f"    [Skip] insufficient samples/events")
        return None

    max_d = np.quantile(aligned[time_col].to_numpy(), grid["q_end"])
    curve_start_d = grid["curve_start_y"] * 365.25
    curve_n       = grid["curve_n"]

    if max_d <= curve_start_d + 60:
        print(f"    [Skip] follow-up too short "
              f"({curve_start_d / 365.25:.1f}y -> {max_d / 365.25:.1f}y)")
        return None

    times_days  = np.linspace(curve_start_d, max_d, curve_n)
    times_years = times_days / 365.25
    print(f"    Grid: {times_years[0]:.1f}-{times_years[-1]:.1f}y, {curve_n} points")

    rows = []
    for label, col in all_scores:
        if col not in valid:
            continue
        res = compute_auc_curve_with_ci(
            aligned, col, times_days, N_BOOT, RANDOM_SEED, N_JOBS
        )
        if res is None:
            continue
        iauc_mean = float(np.nanmean(res["auc"]))
        print(f"      [{label:15s}] mean AUC = {iauc_mean:.4f}")
        for i, ty in enumerate(times_years):
            rows.append(dict(
                Cohort=cohort_name, Score=label, ScoreCol=col,
                time_years=float(ty),
                AUC=float(res["auc"][i]),
                AUC_ci_low=float(res["ci_low"][i]) if res["ci_low"] is not None else np.nan,
                AUC_ci_high=float(res["ci_high"][i]) if res["ci_high"] is not None else np.nan,
                N=n_total, Events=n_events,
            ))
    return pd.DataFrame(rows)


# ========================= Plotting =========================

def draw_auc_curve(pdf, curve_df, cohort, main_label, comparator_labels):
    """Draw AUC(t) curve for one cohort with all scores overlaid."""
    sub = curve_df[curve_df["Cohort"] == cohort]
    if sub.empty:
        return
    score_order = [main_label] + comparator_labels
    scores = [s for s in score_order if s in sub["Score"].unique()]

    fig, ax = plt.subplots(figsize=(8, 5.5))
    for s in scores:
        d = sub[sub["Score"] == s].sort_values("time_years")
        if d.empty:
            continue
        is_main = (s == main_label)
        is_age  = (s == AGE_LABEL)
        if is_main:
            style = dict(color=COLORS.get("BrainVital8"), linewidth=2.8,
                         linestyle="-", zorder=6)
        elif is_age:
            style = dict(color=COLORS[s], linewidth=1.8,
                         linestyle="--", zorder=5, alpha=0.85)
        else:
            style = dict(color=COLORS.get(s, "#888"), linewidth=1.5,
                         linestyle="-", alpha=0.85, zorder=3)
        iAUC_val = d["AUC"].mean()
        ax.plot(d["time_years"], d["AUC"],
                label=f"{s} (iAUC={iAUC_val:.3f})", **style)
        if d["AUC_ci_low"].notna().any():
            ax.fill_between(d["time_years"],
                            d["AUC_ci_low"], d["AUC_ci_high"],
                            color=style["color"], alpha=0.12)

    ax.axhline(0.5, color="gray", linestyle=":", alpha=0.5, linewidth=0.8)
    ax.set_xlabel("Follow-up (Years)", fontsize=11)
    ax.set_ylabel("Cumulative AUC", fontsize=11)
    ax.set_ylim(0.3, 0.95)
    n_info = int(sub["N"].iloc[0]) if "N" in sub.columns else 0
    e_info = int(sub["Events"].iloc[0]) if "Events" in sub.columns else 0
    ax.set_title(f"{cohort}  (n={n_info:,}, events={e_info:,})",
                 fontsize=13, fontweight="bold")
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=8, loc="lower right", framealpha=0.9, ncol=2)
    fig.tight_layout()
    pdf.savefig(fig, bbox_inches="tight")
    plt.close(fig)


# ========================= Run One Analysis Mode =========================

def run_one_curves_analysis(tag, config, df, cohorts):
    """Run AUC curve analysis for one mode (with_age or without_age)."""
    out_dir = OUT_BASE / tag
    fig_dir = out_dir / "figures"
    csv_dir = out_dir / "tables"
    fig_dir.mkdir(parents=True, exist_ok=True)
    csv_dir.mkdir(parents=True, exist_ok=True)

    main_label, main_col = config["main"]
    comparators = [(AGE_LABEL, AGE_COL)] + config["comparators"]
    comparator_labels = [lab for lab, _ in comparators]

    print(f"\n{'=' * 70}")
    print(f"> AUC curves: [{tag}]  {config['description']}")
    print(f"  Main score: {main_label} ({main_col})")
    print(f"  Comparators: {[f'{l}({c})' for l, c in comparators]}")
    print(f"{'=' * 70}")

    all_curve = []
    for cohort in cohorts:
        df_c = df[df[cohort_col] == cohort].copy()
        res = process_cohort_curves(cohort, df_c, main_label, main_col, comparators)
        if res is not None and not res.empty:
            all_curve.append(res)

    if not all_curve:
        print("  No valid results")
        return

    curve_df = pd.concat(all_curve, ignore_index=True)

    # Save CSV
    csv_path = csv_dir / "AUC_curves_long.csv"
    curve_df.to_csv(csv_path, index=False, encoding="utf-8-sig")
    print(f"\n  Saved: {csv_path} ({len(curve_df)} rows)")

    # Draw figures
    pdf_path = fig_dir / "fig_AUC_curves.pdf"
    with PdfPages(pdf_path) as pdf:
        for cohort in cohorts:
            draw_auc_curve(pdf, curve_df, cohort, main_label, comparator_labels)
    print(f"  Saved: {pdf_path}")


# ========================= Main =========================

def main():
    df = pd.read_csv(DATA_PATH, low_memory=False)
    print(f"Loaded data: {len(df)} rows, {df.shape[1]} columns")

    # Collect all needed columns
    all_cols_needed = {AGE_COL, "BrainVital8", time_col, event_col}
    for cfg in ANALYSIS_CONFIGS.values():
        all_cols_needed.add(cfg["main"][1])
        for _, c in cfg["comparators"]:
            all_cols_needed.add(c)
    all_cols_needed.discard("BrainVital8_age")
    df = coerce_numeric(df, list(all_cols_needed))

    df = df.dropna(subset=[time_col, event_col, cohort_col])
    df = df[df[time_col] > 0].copy()
    print(f"After removing missing: {len(df)} rows")

    # Generate BrainVital8_age composite
    bv8_z   = (df["BrainVital8"] - df["BrainVital8"].mean()) / df["BrainVital8"].std()
    age_q   = pd.qcut(df["age"], q=4, labels=False).astype(float)
    age_q_z = (age_q - age_q.mean()) / age_q.std()
    df["BrainVital8_age"] = bv8_z - age_q_z
    print(f"Generated BrainVital8_age: {df['BrainVital8_age'].notna().sum()} non-missing")

    # Auto-detect time unit and convert years to days if needed
    print("\nTime unit detection:")
    for cohort in df[cohort_col].dropna().unique():
        mask     = df[cohort_col] == cohort
        max_time = df.loc[mask, time_col].max()
        if max_time < 50:
            df.loc[mask, time_col] = df.loc[mask, time_col] * 365.25
            print(f"  {cohort}: max={max_time:.2f} -> years, converted to days")
        else:
            print(f"  {cohort}: max={max_time:.2f} -> already in days")

    # Cohort order
    preferred = ["UKB", "ELSA", "SHARE", "HRS", "KLOSA", "CHARLS"]
    cohorts_present = df[cohort_col].unique().tolist()
    cohorts = [c for c in preferred if c in cohorts_present] + \
              sorted([c for c in cohorts_present if c not in preferred])
    print(f"\nCohorts ({len(cohorts)}): {cohorts}")

    for tag, cfg in ANALYSIS_CONFIGS.items():
        run_one_curves_analysis(tag, cfg, df, cohorts)

    print(f"\nDone. Output directory: {OUT_BASE}")


if __name__ == "__main__":
    main()